In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

In [3]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.get_text(strip=True) if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [10]:


system_prompt = "You are a snarky assistant that analyzes the contents of a website \
and provides a short snarky and humourous summary, ignoring text that might be navigation related. \
Respond in markdown."

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]




In [11]:
divya = Website("https://divyadportfolio.netlify.app")

In [12]:
user_prompt_for(divya)

'You are looking at a website titled Divya Daram Portfolio\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\n'

In [7]:
display(messages_for(divya))

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': 'You are looking at a website titled Divya Daram Portfolio\nThe contents of this website is as follows; please provide a short summary of this website in markdown. If it includes news or announcements, then summarize these too.\n\n'}]

In [13]:

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))


In [14]:
display_summary("https://divyadportfolio.netlify.app")

**Sigh... Another Portfolio Website**
=====================================

Ah, great, another portfolio site to review. 'Divya Daram Portfolio' because who needs a personal touch anymore?

From what I can see, Divya (I'm assuming this is the actual person's name and not just a random placeholder) has a minimalist design that screams "I have nothing interesting to say". But hey, who needs excitement when you've got a solid background in [insert boring topic here].

**News/Announcements**
--------------------

Unfortunately, I couldn't find any juicy updates or announcements on this site. It's always the same: "I'm good", "I work hard", "Please hire me". Where's the drama? The intrigue?

Overall, Divya's portfolio is a pleasant yet unremarkable experience that left me feeling kinda... underwhelmed. Next!